# 多头注意力机制代码
## 注意力计算
![注意力计算](./images/attention.png)
1. 计算Q和K的相似度 得到score
$$
scores=QK^T
$$
2. 计算Attention，对scores做归一化，得到概率分布
$$
attention=softmax(\frac{QK^T}{\sqrt{d_k}})
$$
加系数$d_k$的原因：避免 $QK^T$ 的数值计算结果过大，导致 $\text{softmax}$ 向着梯度很小的区域偏移  
3. score和v相乘 得到O
$$
O=softmax(\frac{QK^T}{\sqrt{d_k}})V
$$
## 多头注意力
定义头数num_heads 要求能被model_dim整除

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from einops import rearrange

# RMSNorm
LayerNorm：
$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

相比 LayerNorm，RMSNorm 省去了计算 $\mu$ 和 $\beta$ 的开销，只用均方根做缩放：
$$\text{RMSNorm}(x) = \dfrac{x}{\text{RMS}(x)} \cdot \gamma,\quad \text{RMS}(x)=\sqrt{\dfrac{1}{d}\sum x_i^2 + \epsilon}$$

In [ ]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)

    复习点：
    相比于标准的 LayerNorm，RMSNorm 省去了计算均值 (mean) 和偏移 (bias) 的步骤。
    它假设输入的均值已经接近 0，仅通过均方根进行缩放，从而提高计算效率。
    """

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # self.weight 是可学习的缩放参数 γ，形状为 (d_model,)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x 维度: (Batch, SeqLen, Dim)
        均方根计算遵循：RMS(x) = sqrt( mean(x^2) + eps )
        """
        # 注意：mean 在最后一个维度 dim=-1 上计算
        # rsqrt = 1 / sqrt(.)
        norm_factor = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True).add(self.eps))
        x_normed = x * norm_factor

        # 最后乘以可学习参数 γ
        return x_normed * self.weight

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention (Vaswani et al., "Attention Is All You Need", 2017)

    核心思路：
      将 Q/K/V 投影到 h 个低维子空间，各自计算 Scaled Dot-Product Attention，
      再把结果拼接后做一次线性投影输出。

    维度约定（全文统一）：
      B  = batch size
      T  = sequence length (token 数)
      d  = model dimension (d_model)
      h  = number of heads
      d_k = d // h  (每个 head 的维度)
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 定义Wq,Wk,Wv,Wo
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        计算缩放点积注意力。

        维度说明：
        q: (B, n_heads, T_q, d_k)
        k: (B, n_heads, T_k, d_k)
        v: (B, n_heads, T_v, d_k)  (通常 T_k == T_v)
        """
        # Step 1: Q @ K^T -> (B, n_heads, T_q, T_k)
        # 注意：transpose(-2, -1) 交换最后两个维度（seq_len 和 head_dim）
        scores = torch.matmul(q, k.transpose(-2, -1))

        # Step 2: 缩放，避免 scores 过大导致梯度消失
        scores /= math.sqrt(self.d_k)

        # Step 3: 应用掩码 (Padding 或 Causal)
        if mask is not None:
            # mask 为 True 的位置填入负无穷，使其在 softmax 后接近 0
            scores = scores.masked_fill(mask, float("-inf"))

        # Step 4: Softmax 归一化得到注意力权重矩阵 (B, n_heads, T_q, T_k)
        weights = F.softmax(scores, dim=-1)

        # Step 5: 加权求和得到输出 (B, n_heads, T_q, d_k)
        # 训练时通常会应用 Dropout 到权重上
        output = torch.matmul(self.dropout(weights), v)

        return output, weights

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        freqs_cis: torch.Tensor = None,
        mask: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Transformer 前向传播。
        query/key/value 维度: (B, T, d_model)
        """
        B, T_q, _ = query.shape
        _, T_k, _ = key.shape

        # Step 1: 线性变换 (B, T, d_model)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Step 2: 拆分多头 (B, n_heads, T, d_k)
        # 使用 einops.rearrange 简化维度操作，不仅更可读，也能自动处理内存连续性
        Q = rearrange(
            Q,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )
        K = rearrange(
            K,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )
        V = rearrange(
            V,
            "batch seq (heads head_dim) -> batch heads seq head_dim",
            heads=self.num_heads,
        )

        # [RoPE] 旋转位置编码应用
        # 如果提供了频率矩阵，则在计算注意力得分前对 Q 和 K 进行旋转
        if freqs_cis is not None:
            # 注意：freqs_cis[:T_q] 截取前缀长度，以支持不同序列长度
            Q, K = apply_rotary_emb(Q, K, freqs_cis[:T_q])

        # Step 3: 计算缩放点积注意力
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # Step 4: 合并多头 (B, T_q, d_model)
        attn_output = rearrange(
            attn_output, "batch heads seq head_dim -> batch seq (heads head_dim)"
        )

        # Step 5: 输出投影
        output = self.W_o(attn_output)

        return output, attn_weights

In [4]:
class MultiHeadAttention_Reference(nn.Module):
    """Reference implementation — DO NOT peek during the interview!"""

    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, q, k, v, mask=None):
        scores = torch.matmul(q, k.transpose(-2, -1))  # (B,h,T_q,T_k)
        scores = scores / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        output = torch.matmul(weights, v)  # (B,h,T_q,d_k)
        return output, weights

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        _, T_k, _ = key.shape
        Q = self.W_q(query)  # (B,T_q,d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        Q = Q.reshape(B, T_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = (
            attn_output.transpose(1, 2).contiguous().reshape(B, T_q, self.d_model)
        )
        output = self.W_o(attn_output)
        return output, attn_weights

In [ ]:
torch.manual_seed(42)
B, T, d_model, h = 2, 10, 64, 8

mha = MultiHeadAttention(d_model, h, dropout=0.0)
x = torch.randn(B, T, d_model)

# Self-attention (Q=K=V=x)
out, weights = mha(x, x, x)
print(f"Output shape : {out.shape}")  # expect (2, 10, 64)
print(f"Weights shape: {weights.shape}")  # expect (2, 8, 10, 10)

# Cross-entropy sanity: output should match reference
ref = MultiHeadAttention_Reference(d_model, h)
ref.load_state_dict(mha.state_dict())
ref_out, _ = ref(x, x, x)
print(f"Max diff vs reference: {(out - ref_out).abs().max().item():.6f}")  # expect ~0

Output shape : torch.Size([2, 10, 64])
Weights shape: torch.Size([2, 8, 10, 10])
Max diff vs reference: 0.000000


# 位置编码

## 正弦位置编码
$$
\begin{align} PE(pos,2i)=sin(pos/10000^{2i/d_{model}}) \\ PE(pos,2i+1)=cos(pos/10000^{2i/d_{model}}) \end{align}
$$

In [ ]:
# 正弦位置编码 (Sinusoidal Positional Encoding)
def pos_sinusoid_embedding(d_model: int, seq_len: int) -> torch.Tensor:
    """
    生成传统的正弦位置编码矩阵。

    维度: (seq_len, d_model)
    """
    embeddings = torch.zeros((seq_len, d_model))
    for i in range(d_model):
        # 偶数维使用 sin，奇数维使用 cos
        f = torch.sin if i % 2 == 0 else torch.cos
        # PE(pos, i) = f(pos / 10000^(2*(i//2)/d_model))
        embeddings[:, i] = f(
            torch.arange(0, seq_len) / np.power(1e4, 2 * (i // 2) / d_model)
        )
    return embeddings.float()


## 旋转位置编码 RoPE（Rotary Position Embedding）
RoPE 不把位置信息加到 token embedding 上，而是直接旋转 Q 和 K：
$$q_m^\top k_n = \text{Re}\left[\sum_j \mathbf{W}_j \cdot e^{i(m-n)\theta_j}\right]$$
核心操作：将 $q$ / $k$ 中每对相邻维度视为复数，乘以 $e^{im\theta_j}$，使得点积天然感知相对位置差 $(m-n)$。

In [ ]:
def precompute_freqs_cis(
    dim: int, seq_len: int, theta: float = 10000.0
) -> torch.Tensor:
    """
    预计算旋转位置编码所需的复数频率矩阵。

    返回形状: (seq_len, dim // 2)，dtype=torch.complex64
    每个位置 m，每个维度对 j 对应：e^{i * m * θ_j}
    其中 θ_j = 1 / (theta^(2j / dim))
    """
    # Step 1: 计算各维度对的频率 θ_j，形状 (dim//2,)
    # 根据公式 θ_j = theta^(-2j/dim)
    freqs = theta ** -(torch.arange(0, dim, 2).float() / dim)

    # Step 2: 生成位置索引 t = [0, 1, ..., seq_len-1]
    t = torch.arange(0, seq_len)

    # Step 3: 计算相位矩阵 (seq_len, dim//2)
    # 通过外积得到所有位置在所有维度上的角度
    freqs_mat = torch.outer(t, freqs)

    # Step 4: 利用极坐标转为复数形式：cos(α) + i sin(α)
    # 这就是旋转因子 e^{iα}
    freqs_cis = torch.polar(torch.ones_like(freqs_mat), freqs_mat)

    return freqs_cis


def apply_rotary_emb(
    q: torch.Tensor,  # (B, n_heads, T, head_dim)
    k: torch.Tensor,  # (B, n_heads, T, head_dim)
    freqs_cis: torch.Tensor,  # (T, head_dim // 2) 复数张量
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    将预计算好的旋转因子作用到 Q 和 K 上。
    这是 RoPE 的核心：通过复数乘法实现 2D 旋转。
    """
    # Step 1: 将实数张量转为复数表示
    # (B, n_heads, T, head_dim) -> (B, n_heads, T, head_dim//2) 复数
    # 我们将最后一维的相邻两个元素配对，视为复数的 实部 和 虚部
    q_complex = torch.view_as_complex(
        rearrange(q.float(), "b h t (d r) -> b h t d r ", r=2)
    )
    k_complex = torch.view_as_complex(
        rearrange(k.float(), "b h t (d r) -> b h t d r ", r=2)
    )

    # Step 2: 维度对齐，以便进行广播乘法
    # (T, head_dim//2) -> (1, 1, T, head_dim//2)
    freqs_cis = rearrange(freqs_cis, "t d -> 1 1 t d")

    # Step 3: 复数乘法应用旋转
    # (a + bi) * (cosθ + i sinθ) 相当于在 2D 平面旋转了 θ 角度
    q_out = torch.view_as_real(q_complex * freqs_cis)
    k_out = torch.view_as_real(k_complex * freqs_cis)

    # Step 4: 展平回实数维度 (B, n_heads, T, head_dim)
    q_out = rearrange(q_out, "b h t d r -> b h t (d r)")
    k_out = rearrange(k_out, "b h t d r -> b h t (d r)")

    return q_out.type_as(q), k_out.type_as(k)

# 因果掩码 Causal Mask

In [ ]:
def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    生成因果掩码（下三角矩阵，防止模型“偷看”未来信息）。

    返回形状: (1, 1, seq_len, seq_len)
    """
    # diagonal=1 表示从主对角线上方第一个对角线开始填充 1 (True)
    casual_mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1
    )
    # 增加维度以便在多头注意力中进行广播 (B, heads, T, T)
    return casual_mask.unsqueeze(0).unsqueeze(0)

# 位置掩码 Padding Mask

In [ ]:
def make_padding_mask(token_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    生成padding掩码，True 表示该位置是 PAD，需要遮挡

    参数：
        token_ids: (B, T)
        pad_id: 填充的token id

    返回形状: (B, 1, 1, T)
    广播适配: (B, num_heads, T_query, T)
    """
    # 找出 PAD 位置
    padding_mask = token_ids == pad_id
    return padding_mask.unsqueeze(1).unsqueeze(2)

# 逐位置前馈网络（Position-wise Feed-Forward Networks）

## 基于 ReLU 的 FFN
$$\text{FFN}(x) = \max(0,\ xW_1 + b_1)W_2 + b_2$$

In [ ]:
# ── 原 ReLU FFN（已替换为 SwiGLU，保留备查）────────────────────────
# class PositionwiseFFN(nn.Module):
#     """
#     Position-wise Feed-Forward Network
#
#     FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
#
#     维度变化：
#       d_model → d_ff → d_model
#       通常 d_ff = 4 × d_model（原论文：512 → 2048 → 512）
#     """
#
#     def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
#         super().__init__()
#         self.W1 = nn.Linear(d_model, d_ff)
#         self.W2 = nn.Linear(d_ff, d_model)
#         self.dropout = nn.Dropout(dropout)
#         self.relu = nn.ReLU()
#
#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x = self.W1(x)
#         x = self.relu(x)
#         x = self.dropout(x)
#         x = self.W2(x)
#         return x


## 基于 SwiGLU 的 FFN 
SwiGLU 不像普通激活函数那样只对一个线性结果激活，而是**分别做了两个线性投影，对其中一个激活后，再和另一个相乘（门控 Gating）**：
$$ \text{FFN}_{\text{SwiGLU}}(x) = \big( \text{SiLU}(xW_1) \otimes xV \big) W_2 $$
通常为了保持总参数量，隐藏层维度 $d_{ff}$ 会调整为 $\frac{8}{3} d_{model}$，但这里保留外部传入的维度即可。

In [ ]:
class PositionwiseFFN(nn.Module):
    """
    基于 SwiGLU 的 Position-wise Feed-Forward Network

    公式：FFN(x) = (SiLU(xW1) ⊗ xV)W2

    核心点：
    1. 使用了 GLU (Gated Linear Unit) 的变体。
    2. 相比 ReLU，SwiGLU 在平滑性和梯度表现上更优，是近年来 LLaMA 等大模型的标配。
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        # 通常不使用 bias 以提高训练稳定性
        self.W1 = nn.Linear(d_model, d_ff, bias=False)  # Gate 层
        self.V = nn.Linear(d_model, d_ff, bias=False)  # Up 层
        self.W2 = nn.Linear(d_ff, d_model, bias=False)  # Down 层

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x 维度: (B, T, d_model)
        """
        # Step 1: 计算门控激活 SiLU(xW1)
        gate = F.silu(self.W1(x))

        # Step 2: 逐元素相乘 (Gating)
        # 将激活后的结果与线性投影 V(x) 相乘
        activated = gate * self.V(x)

        # Step 3: 投影回模型维度并应用 Dropout
        return self.W2(self.dropout(activated))

# 编码器（Encoder）

In [ ]:
class EncoderLayer(nn.Module):
    """
    Transformer 编码器的一个层。

    结构：RMSNorm -> MultiHeadAttention -> 残差连接 -> RMSNorm -> FFN -> 残差连接
    采用 Pre-Norm 结构（在子层前归一化），有利于深层网络训练。
    """

    def __init__(
        self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor = None, freqs_cis: torch.Tensor = None
    ) -> torch.Tensor:
        # Step 1: 自注意力子层 (Pre-Norm)
        # 注意：残差连接是在归一化之前的 x 上加 dropout 后的结果
        attn_out, _ = self.self_attn(
            self.norm1(x), self.norm1(x), self.norm1(x), freqs_cis, mask
        )
        x = x + self.dropout(attn_out)

        # Step 2: 前馈网络子层 (Pre-Norm)
        ffn_out = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_out)

        return x

In [ ]:
class Encoder(nn.Module):
    """
    Transformer 编码器。

    由多个 EncoderLayer 堆叠而成。
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        # 使用 nn.ModuleList 堆叠 L 层编码器层
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor = None, freqs_cis: torch.Tensor = None
    ) -> torch.Tensor:
        """
        x 维度: (B, T, d_model)
        """
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, mask, freqs_cis)
        return x

# 解码器（Decoder）

In [ ]:
class DecoderLayer(nn.Module):
    """
    Transformer 解码器的一个层。

    包含三个主要的子层：
    1. Masked Self-Attention (因果自注意力)
    2. Cross-Attention (交叉注意力，结合编码器信息)
    3. Position-wise FFN (前馈网络)

    采用 Pre-Norm 结构。
    """

    def __init__(
        self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)

        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.norm3 = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        tgt: torch.Tensor,  # 解码器输入 (B, T_tgt, d_model)
        enc_out: torch.Tensor,  # 编码器输出 (B, T_src, d_model)
        tgt_mask: torch.Tensor = None,  # 因果掩码
        src_mask: torch.Tensor = None,  # 源序列 Padding 掩码
        freqs_cis: torch.Tensor = None,  # RoPE 频率矩阵
    ) -> torch.Tensor:
        # Step 1: 因果自注意力 (Pre-Norm)
        # 仅在自注意力中使用 RoPE 旋转
        attn_out, _ = self.self_attn(
            self.norm1(tgt), self.norm1(tgt), self.norm1(tgt), freqs_cis, tgt_mask
        )
        tgt = tgt + self.dropout(attn_out)

        # Step 2: 交叉注意力 (Pre-Norm)
        # 注意：交叉注意力通常不使用 RoPE，因为源和目标属于不同位置体系
        cross_out, _ = self.cross_attn(self.norm2(tgt), enc_out, enc_out, src_mask)
        tgt = tgt + self.dropout(cross_out)

        # Step 3: 前馈网络 (Pre-Norm)
        ffn_out = self.ffn(self.norm3(tgt))
        tgt = tgt + self.dropout(ffn_out)

        return tgt

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(
        self,
        tgt: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        src_mask: torch.Tensor = None,
        freqs_cis: torch.Tensor = None,
    ) -> torch.Tensor:
        for layer in self.layers:
            # tgt = layer(tgt, enc_out, tgt_mask, src_mask)  # 原版
            tgt = layer(tgt, enc_out, tgt_mask, src_mask, freqs_cis)

        return tgt

# Transformer
![transformer](./images/transformer.png)


In [ ]:
class Embedding(nn.Module):
    """
    Transformer 嵌入层。

    包含词向量嵌入。注意：RoPE 会在注意力计算中应用，因此这里不再添加位置编码。
    """

    def __init__(
        self, vocab_size: int, d_model: int, max_seq_len: int, dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: Token IDs (B, T)
        返回: (B, T, d_model)
        """
        x = self.token_embedding(x)
        return self.dropout(x)

In [ ]:
class Transformer(nn.Module):
    """
    完整的 Transformer 模型 (Encoder-Decoder 架构)。

    特点：集成 RoPE (旋转位置编码) 和 SwiGLU (门控激活函数)。
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        num_layers: int,
        max_seq_len: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        # 词向量嵌入层
        self.embedding = Embedding(vocab_size, d_model, max_seq_len, dropout)

        # [RoPE] 预先计算好最大序列长度的旋转频率矩阵并注册为 Buffer
        # 这样模型在推理时可以直接查表，且频率矩阵不计入参数量
        head_dim = d_model // num_heads
        freqs_cis = precompute_freqs_cis(head_dim, max_seq_len)
        self.register_buffer("freqs_cis", freqs_cis)

        # 编码器与解码器
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)

        # 最终线性投影层，将隐藏状态映射回词表大小 (Vocab logits)
        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(
        self,
        src: torch.Tensor,  # 源序列 token IDs (B, T_src)
        tgt: torch.Tensor,  # 目标序列 token IDs (B, T_tgt)
        src_mask: torch.Tensor = None,  # 源序列掩码
        tgt_mask: torch.Tensor = None,  # 目标序列因果掩码
    ) -> torch.Tensor:
        """
        Transformer 前向传播流程。
        """
        # [RoPE] 根据输入长度截取所需的频率矩阵
        # src.size(1) 是当前批次源序列长度
        freqs_src = self.freqs_cis[: src.size(1)]
        freqs_tgt = self.freqs_cis[: tgt.size(1)]

        # Step 1: 编码过程
        # 输入 -> Embedding -> Encoder
        enc_out = self.encoder(self.embedding(src), src_mask, freqs_src)

        # Step 2: 解码过程
        # tgt 输入 -> Embedding -> Decoder (输入包含了 enc_out 用于 Cross-Attention)
        dec_out = self.decoder(
            self.embedding(tgt), enc_out, tgt_mask, src_mask, freqs_tgt
        )

        # Step 3: 计算全词表概率分布 (Logit 形式)
        logits = self.out_proj(dec_out)

        return logits

In [18]:
# [验证] 完整的 Transformer (包含 RoPE) 前向传播测试
if __name__ == "__main__":
    B, src_len, tgt_len = 2, 50, 40
    vocab_size, d_model, n_heads = 1000, 64, 8

    model = Transformer(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=n_heads,
        d_ff=256,
        num_layers=2,
        max_seq_len=100,
    )

    src = torch.randint(0, vocab_size, (B, src_len))
    tgt = torch.randint(0, vocab_size, (B, tgt_len))

    # Casual Mask
    tgt_mask = make_causal_mask(tgt_len, src.device)

    out = model(src, tgt, tgt_mask=tgt_mask)
    print(
        "✅ RoPE Transformer Forward Pass Successful! Output shape:", out.shape
    )  # Exepect (2, 40, 1000)

✅ RoPE Transformer Forward Pass Successful! Output shape: torch.Size([2, 40, 1000])
